Q1

In [1]:
import csv

rows = [
    ["이름", "학번", "중간", "기말", "과제"],
    ["김언어", "2026-10000", 85, 92, 90],
    ["이국문", "2026-12345", 78, 88, 85],
    ["박영문", "2026-13579", 95, 90, 100],
    ["최역사", "2025-11111", "", 82, 88],  
]

with open("scores.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

In [6]:
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path:str, json_path:str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
    except FileNotFoundError:
        logging.warning(f"csv 파일이 존재하지 않습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"csv 파일의 인코딩이 잘못되었습니다: {csv_path}")
        return 0
    
    results = []

    for row in rows:
        name: str = row["이름"]
        sid: str = row["학번"]
        mid: int = row["중간"]
        fin: int = row["기말"]
        hw: int = row["과제"]

        if mid == "" or fin == "" or hw == "":
            mid = None if mid == "" else int(mid)
            fin = None if fin == "" else int(fin)
            hw = None if hw == "" else int(hw)
            avg = None
            grade = None
            logging.info(f"{name}: 평균 None, 등급 None (결측값 존재)")
        else:
            mid, fin, hw = int(mid), int(fin), int(hw)
            avg = round(mid*0.3 + fin*0.5 + hw*0.2, 2)
            if avg >= 90:
                grade = "A"
            elif avg >= 80:
                grade = "B"
            elif avg >= 70:
                grade = "C"
            else:
                grade = "F"
            logging.info(f"{name}: 평균 {avg}, 등급 {grade}")
        
        results.append({
            "이름": name,
            "학번": sid,
            "점수": {"중간": mid, "기말": fin, "과제": hw},
            "평균": avg,
            "등급": grade
        })

    with open(json_path, "w", encoding = "utf-8") as f:
        json.dump(results, f, ensure_ascii = False, indent = 4)

    return len(results)

make_report("scores.csv", "report.json")
with open("report.json", "r", encoding="utf-8") as f:
    print(f.read())

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None (결측값 존재)


[
    {
        "이름": "김언어",
        "학번": "2026-10000",
        "점수": {
            "중간": 85,
            "기말": 92,
            "과제": 90
        },
        "평균": 89.5,
        "등급": "B"
    },
    {
        "이름": "이국문",
        "학번": "2026-12345",
        "점수": {
            "중간": 78,
            "기말": 88,
            "과제": 85
        },
        "평균": 84.4,
        "등급": "B"
    },
    {
        "이름": "박영문",
        "학번": "2026-13579",
        "점수": {
            "중간": 95,
            "기말": 90,
            "과제": 100
        },
        "평균": 93.5,
        "등급": "A"
    },
    {
        "이름": "최역사",
        "학번": "2025-11111",
        "점수": {
            "중간": null,
            "기말": 82,
            "과제": 88
        },
        "평균": null,
        "등급": null
    }
]


빈 문자열 여부로 결측값을 감지했다. 세 점수 중 하나라도 빈 칸이면 평균(avg)과 등급(grade)을 모두 None으로 처리하되, 점수 딕셔너리에는 입력된 값은 그대로 기록하고 빈 칸만 None으로 저장했다.

인코딩 선택: 문제에서 CSV가 UTF-8(BOM 없음)임을 명시했으므로 encoding="utf-8"을 사용했다. BOM이 없으므로 utf-8-sig가 아닌 utf-8이 적절하다.

'FileNotFoundError'와 'UnicodeDecodeError'를 각각 따로 잡아 오류 원인을 구체적으로 로그에 기록했다. 

Q2

In [11]:
class InvalidJamoError(ValueError):
    pass

In [13]:
def classify_jamo(c:str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"str 타입이 아닙니다: {type(c)}")
    if len(c) != 1:
        raise ValueError(f"길이가 1이어야 합니다: {repr(c)}")
    if "\u3131" <= c <= "\u314e":
        return "자음"
    if "\u314f" <= c <= "\u3163":
        return "모음"
    raise InvalidJamoError(f"한글 자모가 아닙니다: {repr(c)}")

In [14]:
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result: str = classify_jamo(item)
        print(result)
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1이어야 합니다: 'AB'
[TypeError] str 타입이 아닙니다: <class 'int'>
자음
모음
[ValueError] 길이가 1이어야 합니다: ''


'InvalidJamoError'는 'ValueError'의 자식이므로 except 절에서 'InvalidJamoError'를 'ValueError'보다 앞에 배치했다. 순서가 바뀌면 'InvalidJamoError'가 'ValueError' 단계에서 검출되어 오류의 종류를 구분할 수 없다. 

except Exception을 사용하지 않고 세 예외를 각각 독립적으로 잡아 어떤 종류의 오류가 발생하였는지 정확히 파악할 수 있도록 했다.